# Week 5, Notebook 1: Attention & Transformers from Scratch
## Pure Python + NumPy — The Architecture Behind GPT, BERT, and Modern AI

**What you'll build:** Self-attention, multi-head attention, and a complete Transformer encoder block — all in pure NumPy.

**New concepts:**
- Self-attention: every token attends to every other token
- Query, Key, Value decomposition
- Multi-head attention: parallel attention with different "perspectives"
- Positional encoding: giving the model a sense of order
- The Transformer block: Attention → Add & Norm → FFN → Add & Norm

**Builds on:** All previous weeks + GNN intuition (message passing ≈ attention)

**Time estimate:** 60–75 minutes

---
### The Key Insight
A Transformer is a GNN where **every token is connected to every other token**, and the edge weights (attention) are **learned dynamically** from the data.

```
GNN:  fixed graph structure, learned features
Transformer: learned graph structure (attention), learned features
```

## Part 1: Self-Attention — The Core Mechanism

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

# ============================================================
# Self-attention from absolute zero
# ============================================================
# Input: a sequence of token embeddings
# Output: a sequence of context-aware embeddings
#
# For each token:
#   1. Compute Query (what am I looking for?)
#   2. Compute Key (what do I contain?)
#   3. Compute Value (what information do I carry?)
#   4. Attention score = Query · Key^T / sqrt(d_k)
#   5. Output = softmax(scores) · Values

def softmax(x, axis=-1):
    exp_x = np.exp(x - x.max(axis=axis, keepdims=True))
    return exp_x / exp_x.sum(axis=axis, keepdims=True)


class SelfAttention:
    """Single-head self-attention."""

    def __init__(self, d_model, d_k):
        """
        d_model: input embedding dimension
        d_k: key/query dimension (also value dimension here)
        """
        scale = np.sqrt(2.0 / d_model)
        self.W_Q = np.random.randn(d_model, d_k) * scale  # Query projection
        self.W_K = np.random.randn(d_model, d_k) * scale  # Key projection
        self.W_V = np.random.randn(d_model, d_k) * scale  # Value projection
        self.d_k = d_k

    def forward(self, X):
        """
        X: (seq_len, d_model) — sequence of token embeddings
        Returns: (seq_len, d_k) — context-aware embeddings
        """
        # Project to Q, K, V
        Q = X @ self.W_Q   # (seq_len, d_k) — what each token looks for
        K = X @ self.W_K   # (seq_len, d_k) — what each token advertises
        V = X @ self.W_V   # (seq_len, d_k) — what each token carries

        # Attention scores: how much does token i attend to token j?
        scores = (Q @ K.T) / np.sqrt(self.d_k)  # (seq_len, seq_len)

        # Softmax: normalize to get attention weights
        self.attn_weights = softmax(scores)  # (seq_len, seq_len)

        # Weighted sum of values
        output = self.attn_weights @ V  # (seq_len, d_k)
        return output


# Example: a 5-word sentence
tokens = ["The", "cat", "sat", "on", "mat"]
seq_len = len(tokens)
d_model = 8   # embedding dimension
d_k = 6       # attention dimension

# Random embeddings (in practice, these come from an embedding layer)
X = np.random.randn(seq_len, d_model)

# Run self-attention
attn = SelfAttention(d_model, d_k)
output = attn.forward(X)

print(f"Input shape:  {X.shape}  (seq_len x d_model)")
print(f"Output shape: {output.shape}  (seq_len x d_k)")
print(f"Attention weights shape: {attn.attn_weights.shape}  (seq_len x seq_len)")

# Visualize attention weights
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(attn.attn_weights, cmap='Blues', vmin=0)
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(tokens, fontsize=12)
ax.set_yticklabels(tokens, fontsize=12)
ax.set_xlabel('Attends TO (Keys)')
ax.set_ylabel('Attends FROM (Queries)')
ax.set_title('Self-Attention Weights')

for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, f'{attn.attn_weights[i,j]:.2f}',
               ha='center', va='center', fontsize=9,
               color='white' if attn.attn_weights[i,j] > 0.3 else 'black')

plt.colorbar(im, ax=ax, label='Attention Weight')
plt.tight_layout()
plt.savefig('w5_01_attention.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nEach row shows: how much does that token attend to every other token.")
print("Row sums to 1.0 (softmax). This IS the 'attention' in 'Attention is All You Need'.")

## Part 2: Multi-Head Attention

One attention head captures ONE type of relationship.
**Multiple heads** capture different relationships in parallel.

"Head 1 might learn syntax, Head 2 might learn semantics, Head 3 might learn position."

Then we concatenate all heads and project back.

In [ ]:
# ============================================================
# Multi-Head Attention
# ============================================================

class MultiHeadAttention:
    """Multi-head self-attention mechanism."""

    def __init__(self, d_model, n_heads):
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.d_model = d_model

        # Each head has its own Q, K, V projections
        self.heads = [SelfAttention(d_model, self.d_k) for _ in range(n_heads)]

        # Output projection: concatenated heads back to d_model
        scale = np.sqrt(2.0 / (self.d_k * n_heads))
        self.W_O = np.random.randn(self.d_k * n_heads, d_model) * scale

    def forward(self, X):
        # Run all heads in parallel
        head_outputs = [head.forward(X) for head in self.heads]

        # Concatenate along feature dimension
        concat = np.concatenate(head_outputs, axis=-1)  # (seq_len, d_k * n_heads)

        # Project back to d_model
        output = concat @ self.W_O  # (seq_len, d_model)
        return output


# 4-head attention
mha = MultiHeadAttention(d_model=8, n_heads=4)
mha_output = mha.forward(X)

print(f"Multi-head attention output: {mha_output.shape}")
print(f"Number of heads: {mha.n_heads}, each with d_k = {mha.d_k}")

# Visualize all 4 heads
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for h in range(4):
    weights = mha.heads[h].attn_weights
    axes[h].imshow(weights, cmap='Blues', vmin=0, vmax=weights.max())
    axes[h].set_title(f'Head {h+1}')
    axes[h].set_xticks(range(seq_len))
    axes[h].set_yticks(range(seq_len))
    axes[h].set_xticklabels(tokens, fontsize=9, rotation=45)
    axes[h].set_yticklabels(tokens, fontsize=9)

plt.suptitle('MULTI-HEAD ATTENTION: 4 Different Attention Patterns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('w5_01_multihead.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nEach head learns a DIFFERENT attention pattern!")
print("This is like having 4 different 'perspectives' on the same data.")

## Part 3: The Complete Transformer Block

A full Transformer encoder block:
```
Input → Multi-Head Attention → Add & Norm → FFN → Add & Norm → Output
         ↑_______________________|              ↑________|
              (residual connection)          (residual connection)
```

In [ ]:
# ============================================================
# Layer Normalization
# ============================================================

class LayerNorm:
    """Layer normalization (Ba et al., 2016)."""

    def __init__(self, d_model, eps=1e-6):
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)
        self.eps = eps

    def forward(self, x):
        mean = x.mean(axis=-1, keepdims=True)
        std = x.std(axis=-1, keepdims=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta


# ============================================================
# Feed-Forward Network (position-wise)
# ============================================================

class FeedForward:
    """Position-wise feed-forward network."""

    def __init__(self, d_model, d_ff):
        scale1 = np.sqrt(2.0 / d_model)
        scale2 = np.sqrt(2.0 / d_ff)
        self.W1 = np.random.randn(d_model, d_ff) * scale1
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * scale2
        self.b2 = np.zeros(d_model)

    def forward(self, x):
        h = np.maximum(0, x @ self.W1 + self.b1)  # ReLU
        return h @ self.W2 + self.b2


# ============================================================
# Positional Encoding
# ============================================================

def positional_encoding(seq_len, d_model):
    """Sinusoidal positional encoding (Vaswani et al., 2017)."""
    PE = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            PE[pos, i] = np.sin(pos / 10000 ** (i / d_model))
            if i + 1 < d_model:
                PE[pos, i+1] = np.cos(pos / 10000 ** (i / d_model))
    return PE


# ============================================================
# Complete Transformer Encoder Block
# ============================================================

class TransformerBlock:
    """One Transformer encoder block."""

    def __init__(self, d_model, n_heads, d_ff):
        self.mha = MultiHeadAttention(d_model, n_heads)
        self.norm1 = LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff)
        self.norm2 = LayerNorm(d_model)

    def forward(self, X):
        # Multi-head attention + residual + norm
        attn_out = self.mha.forward(X)
        X = self.norm1.forward(X + attn_out)  # Residual connection

        # Feed-forward + residual + norm
        ff_out = self.ff.forward(X)
        X = self.norm2.forward(X + ff_out)    # Residual connection

        return X


# Build a 2-block Transformer encoder
d_model = 16
n_heads = 4
d_ff = 64
seq_len = 8

# Random input embeddings + positional encoding
X_seq = np.random.randn(seq_len, d_model) * 0.5
PE = positional_encoding(seq_len, d_model)
X_input = X_seq + PE

# Stack 2 Transformer blocks
block1 = TransformerBlock(d_model, n_heads, d_ff)
block2 = TransformerBlock(d_model, n_heads, d_ff)

h = block1.forward(X_input)
h = block2.forward(h)

print(f"Input:  {X_input.shape}  (seq_len x d_model)")
print(f"Output: {h.shape}  (seq_len x d_model)")
print(f"\nTransformer block components:")
print(f"  Multi-Head Attention: {n_heads} heads, d_k = {d_model // n_heads}")
print(f"  Feed-Forward: {d_model} -> {d_ff} -> {d_model}")
print(f"  Positional Encoding: sinusoidal")
print(f"\nTotal blocks: 2 (each token has seen the full sequence TWICE)")

In [ ]:
# Visualize positional encoding
PE_vis = positional_encoding(50, 32)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(PE_vis.T, cmap='RdBu_r', aspect='auto')
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Dimension')
axes[0].set_title('Positional Encoding Matrix')

for dim in [0, 1, 2, 3, 6, 10]:
    axes[1].plot(PE_vis[:, dim], label=f'dim {dim}', alpha=0.8)
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Value')
axes[1].set_title('PE by Dimension (sinusoidal waves)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.2)

plt.suptitle('POSITIONAL ENCODING: Giving Order to the Transformer', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('w5_01_positional.png', dpi=100, bbox_inches='tight')
plt.show()

print("Lower dimensions = high frequency (position changes fast)")
print("Higher dimensions = low frequency (captures longer-range position)")
print("This lets the model learn both local and global position relationships.")

## ✅ Self-Check

- [ ] You can explain attention in one sentence: "Each token computes a weighted average of all tokens' values, weighted by query-key similarity"
- [ ] You understand Q, K, V: Query = "what I want", Key = "what I offer", Value = "what I carry"
- [ ] You can explain multi-head: "parallel attention heads capture different relationship types"
- [ ] You know why positional encoding exists: attention is permutation-invariant, PE breaks symmetry
- [ ] You see the connection: GNN message passing is attention with a fixed graph; Transformers learn the graph

## ➡️ Next: `W5_02_PyTorch_Transformer.ipynb` — Build a real Transformer in PyTorch